<a href="https://colab.research.google.com/github/Botatopiece9/kpneumo-amr-prediction/blob/main/03_build_gene_table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/kpneumo-amr-prediction"
RESULTS_DIR = f"{PROJECT_DIR}/amrfinder_results"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
files = [f for f in os.listdir(RESULTS_DIR) if f.endswith(".tsv")]
genome_ids = [f.replace(".tsv", "") for f in files]

tables = []
for f in files:
    t = pd.read_csv(f"{RESULTS_DIR}/{f}", sep="\t", dtype={"Name": str})
    tables.append(t)

hits = pd.concat(tables, ignore_index=True)
print("Genomes scanned so far:", len(genome_ids))
print("Total genes/mutations found:", len(hits))

Genomes scanned so far: 200
Total genes/mutations found: 3130


In [ ]:
matrix = pd.crosstab(hits["Name"], hits["Element symbol"])
matrix = (matrix > 0).astype(int)
matrix = matrix.reindex(genome_ids, fill_value=0)
matrix.index.name = "genome_id"

print(matrix.shape)
matrix.iloc[:5, :8]

(200, 219)


Element symbol,aac(3)-IId,aac(3)-IIe,aac(3)-IVa,aac(3)-Ia,aac(3)-VIa,aac(6')-IIa,aac(6')-Ib,aac(6')-Ib'
genome_id,,,,,,,,
573.14212,0,1,0,0,0,0,1,0
573.21765,0,0,0,0,0,0,1,0
573.14819,0,0,0,0,0,0,1,0
573.13171,0,1,0,0,0,0,0,0
573.13326,0,0,0,0,0,0,0,0


In [ ]:
info = pd.read_csv(f"{PROJECT_DIR}/scan_list_200.csv", dtype={"genome_id": str})
info = info[["genome_id", "label", "mlst"]]

table = info.merge(matrix, left_on="genome_id", right_index=True, how="inner")

print(table.shape)
print(table["label"].value_counts())

(200, 222)
label
Resistant      100
Susceptible    100
Name: count, dtype: int64


In [ ]:
prevalence = matrix.mean().sort_values(ascending=False)
print(prevalence.head(15).round(2).to_string())
print()
print("Genes in every genome:", (prevalence == 1).sum())
print("Genes in only one genome:", (prevalence * len(matrix) == 1).sum())

Element symbol
fosA              0.96
oqxA              0.87
parC_S80I         0.56
oqxB              0.53
sul1              0.49
qacEdelta1        0.47
blaTEM-1          0.46
gyrA_S83I         0.44
blaCTX-M-15       0.36
blaSHV-11         0.36
aadA2             0.30
aac(6')-Ib-cr5    0.30
sul2              0.29
aph(6)-Id         0.28
aph(3'')-Ib       0.28

Genes in every genome: 0
Genes in only one genome: 86


In [47]:
carb_genes = hits.loc[hits["Subclass"] == "CARBAPENEM", "Element symbol"].unique().tolist()
print("Carbapenem-related elements found:", carb_genes)

table["has_carbapenem_element"] = table[carb_genes].max(axis=1)
print(pd.crosstab(table["label"], table["has_carbapenem_element"]))

Carbapenem-related elements found: ['ompK35_G208VfsTer5', 'ompK36_D135DGD', 'ompK35_E42RfsTer47', 'blaKPC-3', 'blaKPC-2', 'blaOXA-48', 'ompK35_G62AfsTer1', 'ompK36_Q313Ter', 'ompK35_R184AfsTer69', 'ompK35_P12WfsTer1', 'ompK36_T136TDT', 'blaOXA-181', 'ompK35_E132K', 'ompK35_I25KfsTer39', 'blaNDM-5', 'ompK35_R60LfsTer31', 'blaNDM-1', 'ompK35_P255QfsTer5', 'blaNDM-4', 'ompK35_N29TfsTer34', 'ompK35_A59GfsTer21', 'ompK35_F300LfsTer25', 'blaVIM-19', 'ompK35_S110YfsTer35', 'ompK36_Q54RfsTer34', 'ompK35_P12SfsTer10', 'ompK35_N177TfsTer76', 'blaOXA-232', 'blaNDM-7', 'ompK35_V10GfsTer6', 'ompK36_D135DD', 'blaKPC', 'ompK35_K176Ter', 'ompK35_W230Ter', 'ompK35_W316Ter', 'ompK36_W125Ter']
has_carbapenem_element   0    1
label                          
Resistant                0  100
Susceptible             71   29


In [48]:
table.to_csv(f"{PROJECT_DIR}/gene_table_200.csv", index=False)
print("Saved!")

Saved!


In [49]:
drive.flush_and_unmount()
print("All files uploaded to Drive")

All files uploaded to Drive
